# Azure Databricks Final Assessment

## Domain: Hospital Operations Analytics

## Objective

Build an end-to-end Databricks pipeline using:

* PySpark DataFrames
* Spark SQL
* Delta Lake
* CRUD
* MERGE / UPSERT
* History
* Time Travel
* VACUUM
* Parquet to Delta
* Incremental Load
* DLT
* Unity Catalog
* Governance

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Azure Assessment 1").getOrCreate()



---

# Part 0 — Prepare Bigger Dataset

```python
patients_data = [
(1001,"Aarav Khan","Hyderabad",29,"Male","Gold","2023-01-10"),
(1002,"Priya Reddy","Bengaluru",34,"Female","Silver","2023-01-15"),
(1003,"Rahul Mehta","Mumbai",41,"Male","Gold","2023-02-02"),
(1004,"Sneha Kapoor","Delhi",26,"Female","Bronze","2023-02-18"),
(1005,"Kiran Patel","Ahmedabad",37,"Male","Silver","2023-03-01"),
(1006,"Ananya Das","Kolkata",31,"Female","Gold","2023-03-12"),
(1007,"Vikram Singh","Chennai",45,"Male","Bronze","2023-03-20"),
(1008,"Meera Nair","Kochi",28,"Female","Silver","2023-04-05"),
(1009,"Farhan Ali","Hyderabad",39,"Male","Gold","2023-04-15"),
(1010,"Divya Menon","Bengaluru",33,"Female","Silver","2023-04-21"),
(1011,"Arjun Iyer","Chennai",52,"Male","Gold","2023-05-01"),
(1012,"Neha Gupta","Delhi",25,"Female","Bronze","2023-05-11"),
(1013,"Sanjay Rao","Mumbai",48,"Male","Silver","2023-05-19"),
(1014,"Kavya Sharma","Hyderabad",30,"Female","Gold","2023-06-01"),
(1015,"Nikhil Verma","Pune",36,"Male","Silver","2023-06-12"),
(1016,"Ayesha Khan","Kolkata",27,"Female","Bronze","2023-06-20"),
(1017,"Manish Yadav","Lucknow",44,"Male","Gold","2023-07-05"),
(1018,"Pooja Shah","Ahmedabad",32,"Female","Silver","2023-07-18"),
(1019,"Rohan Nair","Kochi",40,"Male","Gold","2023-08-01"),
(1020,"Lakshmi Rao","Chennai",35,"Female","Silver","2023-08-14")
]

patients_columns = [
"patient_id","patient_name","city","age","gender","membership","registration_date"
]

patients_df = spark.createDataFrame(patients_data, patients_columns)
```

---

```python
doctors_data = [
(201,"Dr Sameer Sharma","Cardiology","Hyderabad",1200),
(202,"Dr Kavita Iyer","Dermatology","Bengaluru",800),
(203,"Dr Imran Khan","Orthopedics","Mumbai",1500),
(204,"Dr Ramesh Reddy","Pediatrics","Delhi",900),
(205,"Dr Anita Mehta","Neurology","Hyderabad",2000),
(206,"Dr Joseph Mathew","Cardiology","Chennai",1300),
(207,"Dr Fatima Ali","Dermatology","Kolkata",850),
(208,"Dr Arvind Rao","Orthopedics","Bengaluru",1400),
(209,"Dr Leela Nair","Neurology","Kochi",1900),
(210,"Dr Ganesh Patil","General Medicine","Pune",700)
]

doctors_columns = [
"doctor_id","doctor_name","specialization","doctor_city","consultation_fee"
]

doctors_df = spark.createDataFrame(doctors_data, doctors_columns)
```

---

```python
visits_data = [
(1,1001,201,"2024-03-01","Completed",2),
(2,1002,202,"2024-03-01","Completed",1),
(3,1003,203,"2024-03-02","Completed",3),
(4,1004,204,"2024-03-02","Pending",1),
(5,1005,206,"2024-03-03","Completed",2),
(6,1006,205,"2024-03-03","Completed",4),
(7,1007,207,"2024-03-04","Cancelled",1),
(8,1008,208,"2024-03-04","Completed",2),
(9,1009,201,"2024-03-05","Completed",1),
(10,1010,202,"2024-03-05","Completed",2),
(11,1011,205,"2024-03-06","Pending",3),
(12,1012,204,"2024-03-06","Completed",1),
(13,1013,203,"2024-03-07","Completed",2),
(14,1014,201,"2024-03-07","Completed",3),
(15,1015,210,"2024-03-08","Completed",1),
(16,1016,207,"2024-03-08","Cancelled",2),
(17,1017,209,"2024-03-09","Completed",4),
(18,1018,206,"2024-03-09","Completed",2),
(19,1019,209,"2024-03-10","Completed",3),
(20,1020,206,"2024-03-10","Pending",2),
(21,1001,205,"2024-03-11","Completed",3),
(22,1003,208,"2024-03-11","Completed",2),
(23,1006,201,"2024-03-12","Completed",1),
(24,1009,210,"2024-03-12","Completed",2),
(25,1014,202,"2024-03-13","Completed",1)
]

visits_columns = [
"visit_id","patient_id","doctor_id","visit_date","visit_status","tests_count"
]

visits_df = spark.createDataFrame(visits_data, visits_columns)
```

---

```python
payments_data = [
(301,1,5200,"UPI","Paid"),
(302,2,2800,"Credit Card","Paid"),
(303,3,7500,"Cash","Paid"),
(304,4,2900,"UPI","Pending"),
(305,5,5300,"Debit Card","Paid"),
(306,6,10000,"Credit Card","Paid"),
(307,7,2850,"Cash","Cancelled"),
(308,8,5400,"UPI","Paid"),
(309,9,3200,"UPI","Paid"),
(310,10,4800,"Credit Card","Paid"),
(311,11,8000,"UPI","Pending"),
(312,12,2900,"Cash","Paid"),
(313,13,5500,"Credit Card","Paid"),
(314,14,7200,"UPI","Paid"),
(315,15,2700,"Debit Card","Paid"),
(316,16,4850,"Cash","Cancelled"),
(317,17,9900,"Credit Card","Paid"),
(318,18,5300,"UPI","Paid"),
(319,19,7900,"Debit Card","Paid"),
(320,20,5300,"UPI","Pending"),
(321,21,8000,"UPI","Paid"),
(322,22,5400,"Credit Card","Paid"),
(323,23,3200,"Cash","Paid"),
(324,24,4700,"UPI","Paid"),
(325,25,2800,"UPI","Paid")
]

payments_columns = [
"payment_id","visit_id","bill_amount","payment_mode","payment_status"
]

payments_df = spark.createDataFrame(payments_data, payments_columns)
```

---


In [0]:
patients_data = [
(1001,"Aarav Khan","Hyderabad",29,"Male","Gold","2023-01-10"),
(1002,"Priya Reddy","Bengaluru",34,"Female","Silver","2023-01-15"),
(1003,"Rahul Mehta","Mumbai",41,"Male","Gold","2023-02-02"),
(1004,"Sneha Kapoor","Delhi",26,"Female","Bronze","2023-02-18"),
(1005,"Kiran Patel","Ahmedabad",37,"Male","Silver","2023-03-01"),
(1006,"Ananya Das","Kolkata",31,"Female","Gold","2023-03-12"),
(1007,"Vikram Singh","Chennai",45,"Male","Bronze","2023-03-20"),
(1008,"Meera Nair","Kochi",28,"Female","Silver","2023-04-05"),
(1009,"Farhan Ali","Hyderabad",39,"Male","Gold","2023-04-15"),
(1010,"Divya Menon","Bengaluru",33,"Female","Silver","2023-04-21"),
(1011,"Arjun Iyer","Chennai",52,"Male","Gold","2023-05-01"),
(1012,"Neha Gupta","Delhi",25,"Female","Bronze","2023-05-11"),
(1013,"Sanjay Rao","Mumbai",48,"Male","Silver","2023-05-19"),
(1014,"Kavya Sharma","Hyderabad",30,"Female","Gold","2023-06-01"),
(1015,"Nikhil Verma","Pune",36,"Male","Silver","2023-06-12"),
(1016,"Ayesha Khan","Kolkata",27,"Female","Bronze","2023-06-20"),
(1017,"Manish Yadav","Lucknow",44,"Male","Gold","2023-07-05"),
(1018,"Pooja Shah","Ahmedabad",32,"Female","Silver","2023-07-18"),
(1019,"Rohan Nair","Kochi",40,"Male","Gold","2023-08-01"),
(1020,"Lakshmi Rao","Chennai",35,"Female","Silver","2023-08-14")
]

patients_columns = [
"patient_id","patient_name","city","age","gender","membership","registration_date"
]

patients_df = spark.createDataFrame(patients_data, patients_columns)
doctors_data = [
(201,"Dr Sameer Sharma","Cardiology","Hyderabad",1200),
(202,"Dr Kavita Iyer","Dermatology","Bengaluru",800),
(203,"Dr Imran Khan","Orthopedics","Mumbai",1500),
(204,"Dr Ramesh Reddy","Pediatrics","Delhi",900),
(205,"Dr Anita Mehta","Neurology","Hyderabad",2000),
(206,"Dr Joseph Mathew","Cardiology","Chennai",1300),
(207,"Dr Fatima Ali","Dermatology","Kolkata",850),
(208,"Dr Arvind Rao","Orthopedics","Bengaluru",1400),
(209,"Dr Leela Nair","Neurology","Kochi",1900),
(210,"Dr Ganesh Patil","General Medicine","Pune",700)
]

doctors_columns = [
"doctor_id","doctor_name","specialization","doctor_city","consultation_fee"
]

doctors_df = spark.createDataFrame(doctors_data, doctors_columns)
visits_data = [
(1,1001,201,"2024-03-01","Completed",2),
(2,1002,202,"2024-03-01","Completed",1),
(3,1003,203,"2024-03-02","Completed",3),
(4,1004,204,"2024-03-02","Pending",1),
(5,1005,206,"2024-03-03","Completed",2),
(6,1006,205,"2024-03-03","Completed",4),
(7,1007,207,"2024-03-04","Cancelled",1),
(8,1008,208,"2024-03-04","Completed",2),
(9,1009,201,"2024-03-05","Completed",1),
(10,1010,202,"2024-03-05","Completed",2),
(11,1011,205,"2024-03-06","Pending",3),
(12,1012,204,"2024-03-06","Completed",1),
(13,1013,203,"2024-03-07","Completed",2),
(14,1014,201,"2024-03-07","Completed",3),
(15,1015,210,"2024-03-08","Completed",1),
(16,1016,207,"2024-03-08","Cancelled",2),
(17,1017,209,"2024-03-09","Completed",4),
(18,1018,206,"2024-03-09","Completed",2),
(19,1019,209,"2024-03-10","Completed",3),
(20,1020,206,"2024-03-10","Pending",2),
(21,1001,205,"2024-03-11","Completed",3),
(22,1003,208,"2024-03-11","Completed",2),
(23,1006,201,"2024-03-12","Completed",1),
(24,1009,210,"2024-03-12","Completed",2),
(25,1014,202,"2024-03-13","Completed",1)
]

visits_columns = [
"visit_id","patient_id","doctor_id","visit_date","visit_status","tests_count"
]

visits_df = spark.createDataFrame(visits_data, visits_columns)
payments_data = [
(301,1,5200,"UPI","Paid"),
(302,2,2800,"Credit Card","Paid"),
(303,3,7500,"Cash","Paid"),
(304,4,2900,"UPI","Pending"),
(305,5,5300,"Debit Card","Paid"),
(306,6,10000,"Credit Card","Paid"),
(307,7,2850,"Cash","Cancelled"),
(308,8,5400,"UPI","Paid"),
(309,9,3200,"UPI","Paid"),
(310,10,4800,"Credit Card","Paid"),
(311,11,8000,"UPI","Pending"),
(312,12,2900,"Cash","Paid"),
(313,13,5500,"Credit Card","Paid"),
(314,14,7200,"UPI","Paid"),
(315,15,2700,"Debit Card","Paid"),
(316,16,4850,"Cash","Cancelled"),
(317,17,9900,"Credit Card","Paid"),
(318,18,5300,"UPI","Paid"),
(319,19,7900,"Debit Card","Paid"),
(320,20,5300,"UPI","Pending"),
(321,21,8000,"UPI","Paid"),
(322,22,5400,"Credit Card","Paid"),
(323,23,3200,"Cash","Paid"),
(324,24,4700,"UPI","Paid"),
(325,25,2800,"UPI","Paid")
]

payments_columns = [
"payment_id","visit_id","bill_amount","payment_mode","payment_status"
]

payments_df = spark.createDataFrame(payments_data, payments_columns)

---

# Assessment Tasks

# Part 1 — DataFrame Fundamentals

1. Display all four DataFrames.
2. Print schema for each DataFrame.
3. Count records in each DataFrame.
4. Display first 10 patient records.
5. Display only patient name, city, membership, and age.
6. Display only doctor name, specialization, and consultation fee.
7. Display all visits with status Completed.
8. Display all pending visits.
9. Display patients from Hyderabad and Bengaluru.
10. Display doctors whose consultation fee is greater than 1000.

---


In [0]:
patients_df.show()
doctors_df.show()
visits_df.show()
payments_df.show()

+----------+------------+---------+---+------+----------+-----------------+
|patient_id|patient_name|     city|age|gender|membership|registration_date|
+----------+------------+---------+---+------+----------+-----------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|
|      1007|Vikram Singh|  Chennai| 45|  Male|    Bronze|       2023-03-20|
|      1008|  Meera Nair|    Kochi| 28|Female|    Silver|       2023-04-05|
|      1009|  Farhan Ali|Hyderabad| 39|  Male|      Gold|       2023-04-15|
|      1010| Divya Menon|Bengaluru| 33|Female|    Silver|       2023-04-21|
|      1011|

In [0]:
patients_df.printSchema()
doctors_df.printSchema()
visits_df.printSchema()
payments_df.printSchema()

root
 |-- patient_id: long (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- gender: string (nullable = true)
 |-- membership: string (nullable = true)
 |-- registration_date: string (nullable = true)

root
 |-- doctor_id: long (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- doctor_city: string (nullable = true)
 |-- consultation_fee: long (nullable = true)

root
 |-- visit_id: long (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- doctor_id: long (nullable = true)
 |-- visit_date: string (nullable = true)
 |-- visit_status: string (nullable = true)
 |-- tests_count: long (nullable = true)

root
 |-- payment_id: long (nullable = true)
 |-- visit_id: long (nullable = true)
 |-- bill_amount: long (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)



In [0]:
patients_df.count()
doctors_df.count()
visits_df.count()
payments_df.count()

25

In [0]:
patients_df.show(10)

+----------+------------+---------+---+------+----------+-----------------+
|patient_id|patient_name|     city|age|gender|membership|registration_date|
+----------+------------+---------+---+------+----------+-----------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|
|      1007|Vikram Singh|  Chennai| 45|  Male|    Bronze|       2023-03-20|
|      1008|  Meera Nair|    Kochi| 28|Female|    Silver|       2023-04-05|
|      1009|  Farhan Ali|Hyderabad| 39|  Male|      Gold|       2023-04-15|
|      1010| Divya Menon|Bengaluru| 33|Female|    Silver|       2023-04-21|
+----------+

In [0]:
patients_df.select(patients_df.patient_name,patients_df.city,patients_df.membership,patients_df.age).show()

+------------+---------+----------+---+
|patient_name|     city|membership|age|
+------------+---------+----------+---+
|  Aarav Khan|Hyderabad|      Gold| 29|
| Priya Reddy|Bengaluru|    Silver| 34|
| Rahul Mehta|   Mumbai|      Gold| 41|
|Sneha Kapoor|    Delhi|    Bronze| 26|
| Kiran Patel|Ahmedabad|    Silver| 37|
|  Ananya Das|  Kolkata|      Gold| 31|
|Vikram Singh|  Chennai|    Bronze| 45|
|  Meera Nair|    Kochi|    Silver| 28|
|  Farhan Ali|Hyderabad|      Gold| 39|
| Divya Menon|Bengaluru|    Silver| 33|
|  Arjun Iyer|  Chennai|      Gold| 52|
|  Neha Gupta|    Delhi|    Bronze| 25|
|  Sanjay Rao|   Mumbai|    Silver| 48|
|Kavya Sharma|Hyderabad|      Gold| 30|
|Nikhil Verma|     Pune|    Silver| 36|
| Ayesha Khan|  Kolkata|    Bronze| 27|
|Manish Yadav|  Lucknow|      Gold| 44|
|  Pooja Shah|Ahmedabad|    Silver| 32|
|  Rohan Nair|    Kochi|      Gold| 40|
| Lakshmi Rao|  Chennai|    Silver| 35|
+------------+---------+----------+---+



In [0]:
doctors_df.select(doctors_df.doctor_name,doctors_df.specialization,doctors_df.consultation_fee).show()

+----------------+----------------+----------------+
|     doctor_name|  specialization|consultation_fee|
+----------------+----------------+----------------+
|Dr Sameer Sharma|      Cardiology|            1200|
|  Dr Kavita Iyer|     Dermatology|             800|
|   Dr Imran Khan|     Orthopedics|            1500|
| Dr Ramesh Reddy|      Pediatrics|             900|
|  Dr Anita Mehta|       Neurology|            2000|
|Dr Joseph Mathew|      Cardiology|            1300|
|   Dr Fatima Ali|     Dermatology|             850|
|   Dr Arvind Rao|     Orthopedics|            1400|
|   Dr Leela Nair|       Neurology|            1900|
| Dr Ganesh Patil|General Medicine|             700|
+----------------+----------------+----------------+



In [0]:
visits_df.filter(visits_df.visit_status == "Completed").show()

+--------+----------+---------+----------+------------+-----------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|
+--------+----------+---------+----------+------------+-----------+
|       1|      1001|      201|2024-03-01|   Completed|          2|
|       2|      1002|      202|2024-03-01|   Completed|          1|
|       3|      1003|      203|2024-03-02|   Completed|          3|
|       5|      1005|      206|2024-03-03|   Completed|          2|
|       6|      1006|      205|2024-03-03|   Completed|          4|
|       8|      1008|      208|2024-03-04|   Completed|          2|
|       9|      1009|      201|2024-03-05|   Completed|          1|
|      10|      1010|      202|2024-03-05|   Completed|          2|
|      12|      1012|      204|2024-03-06|   Completed|          1|
|      13|      1013|      203|2024-03-07|   Completed|          2|
|      14|      1014|      201|2024-03-07|   Completed|          3|
|      15|      1015|      210|2024-03-08|   Com

In [0]:
visits_df.filter(visits_df.visit_status=="Pending").show()

+--------+----------+---------+----------+------------+-----------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|
+--------+----------+---------+----------+------------+-----------+
|       4|      1004|      204|2024-03-02|     Pending|          1|
|      11|      1011|      205|2024-03-06|     Pending|          3|
|      20|      1020|      206|2024-03-10|     Pending|          2|
+--------+----------+---------+----------+------------+-----------+



In [0]:
patients_df.filter(patients_df.city.isin("Bengaluru","Hyderabad")).show()

+----------+------------+---------+---+------+----------+-----------------+
|patient_id|patient_name|     city|age|gender|membership|registration_date|
+----------+------------+---------+---+------+----------+-----------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|
|      1009|  Farhan Ali|Hyderabad| 39|  Male|      Gold|       2023-04-15|
|      1010| Divya Menon|Bengaluru| 33|Female|    Silver|       2023-04-21|
|      1014|Kavya Sharma|Hyderabad| 30|Female|      Gold|       2023-06-01|
+----------+------------+---------+---+------+----------+-----------------+



In [0]:
doctors_df.filter(doctors_df.consultation_fee>1000).show()

+---------+----------------+--------------+-----------+----------------+
|doctor_id|     doctor_name|specialization|doctor_city|consultation_fee|
+---------+----------------+--------------+-----------+----------------+
|      201|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|
|      203|   Dr Imran Khan|   Orthopedics|     Mumbai|            1500|
|      205|  Dr Anita Mehta|     Neurology|  Hyderabad|            2000|
|      206|Dr Joseph Mathew|    Cardiology|    Chennai|            1300|
|      208|   Dr Arvind Rao|   Orthopedics|  Bengaluru|            1400|
|      209|   Dr Leela Nair|     Neurology|      Kochi|            1900|
+---------+----------------+--------------+-----------+----------------+



---

# Part 2 — DataFrame Transformations

11. Convert registration_date into date type.
12. Convert visit_date into date type.
13. Add a column test_cost.
14. Add a column estimated_total_bill.
15. Create age_category.
16. Create membership_priority.
17. Create visit_priority.
18. Create high_bill_flag.
19. Rename patient_name to full_name.
20. Drop any temporary column created during transformation.

---

In [0]:
from pyspark.sql.functions import *

In [0]:
patients_df.withColumn("registeration_date",to_date(patients_df.registration_date)).show()


+----------+------------+---------+---+------+----------+-----------------+------------------+
|patient_id|patient_name|     city|age|gender|membership|registration_date|registeration_date|
+----------+------------+---------+---+------+----------+-----------------+------------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|        2023-01-10|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|        2023-01-15|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|        2023-02-02|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|        2023-02-18|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|        2023-03-01|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|        2023-03-12|
|      1007|Vikram Singh|  Chennai| 45|  Male|    Bronze|       2023-03-20|        2023-03-20|
|      1008|  Meera Nair|    Kochi| 28|Female|    

In [0]:
visits_df.withColumn("visit_date",to_date(visits_df.visit_date)).show()

+--------+----------+---------+----------+------------+-----------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|
+--------+----------+---------+----------+------------+-----------+
|       1|      1001|      201|2024-03-01|   Completed|          2|
|       2|      1002|      202|2024-03-01|   Completed|          1|
|       3|      1003|      203|2024-03-02|   Completed|          3|
|       4|      1004|      204|2024-03-02|     Pending|          1|
|       5|      1005|      206|2024-03-03|   Completed|          2|
|       6|      1006|      205|2024-03-03|   Completed|          4|
|       7|      1007|      207|2024-03-04|   Cancelled|          1|
|       8|      1008|      208|2024-03-04|   Completed|          2|
|       9|      1009|      201|2024-03-05|   Completed|          1|
|      10|      1010|      202|2024-03-05|   Completed|          2|
|      11|      1011|      205|2024-03-06|     Pending|          3|
|      12|      1012|      204|2024-03-06|   Com

In [0]:
visits_df=visits_df.withColumn("test_cost",visits_df.tests_count*500)
visits_df.show()

+--------+----------+---------+----------+------------+-----------+---------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|test_cost|
+--------+----------+---------+----------+------------+-----------+---------+
|       1|      1001|      201|2024-03-01|   Completed|          2|     1000|
|       2|      1002|      202|2024-03-01|   Completed|          1|      500|
|       3|      1003|      203|2024-03-02|   Completed|          3|     1500|
|       4|      1004|      204|2024-03-02|     Pending|          1|      500|
|       5|      1005|      206|2024-03-03|   Completed|          2|     1000|
|       6|      1006|      205|2024-03-03|   Completed|          4|     2000|
|       7|      1007|      207|2024-03-04|   Cancelled|          1|      500|
|       8|      1008|      208|2024-03-04|   Completed|          2|     1000|
|       9|      1009|      201|2024-03-05|   Completed|          1|      500|
|      10|      1010|      202|2024-03-05|   Completed|         

In [0]:
visits_doctors_df = visits_df.join(
    doctors_df,
    "doctor_id"
)

visits_doctors_df = visits_doctors_df.withColumn(
    "estimated_total_bill",
    col("consultation_fee") + col("test_cost")
)
visits_doctors_df.show()

+---------+--------+----------+----------+------------+-----------+---------+----------------+----------------+-----------+----------------+--------------------+
|doctor_id|visit_id|patient_id|visit_date|visit_status|tests_count|test_cost|     doctor_name|  specialization|doctor_city|consultation_fee|estimated_total_bill|
+---------+--------+----------+----------+------------+-----------+---------+----------------+----------------+-----------+----------------+--------------------+
|      201|       1|      1001|2024-03-01|   Completed|          2|     1000|Dr Sameer Sharma|      Cardiology|  Hyderabad|            1200|                2200|
|      202|       2|      1002|2024-03-01|   Completed|          1|      500|  Dr Kavita Iyer|     Dermatology|  Bengaluru|             800|                1300|
|      203|       3|      1003|2024-03-02|   Completed|          3|     1500|   Dr Imran Khan|     Orthopedics|     Mumbai|            1500|                3000|
|      204|       4|      10

In [0]:
patients_df.withColumn("age_category",when(patients_df.age<35,"Young").when(patients_df.age<70,"Middle Age").otherwise("Old")).show()

+----------+------------+---------+---+------+----------+-----------------+------------+
|patient_id|patient_name|     city|age|gender|membership|registration_date|age_category|
+----------+------------+---------+---+------+----------+-----------------+------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|       Young|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|       Young|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|  Middle Age|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|       Young|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|  Middle Age|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|       Young|
|      1007|Vikram Singh|  Chennai| 45|  Male|    Bronze|       2023-03-20|  Middle Age|
|      1008|  Meera Nair|    Kochi| 28|Female|    Silver|       2023-04-05|       Young|
|      1009|  Farhan 

In [0]:
patients_df.withColumn("membership_priority",when(patients_df.membership=="Gold",1).when(patients_df.membership=="Silver",2).otherwise(3)).show()

+----------+------------+---------+---+------+----------+-----------------+-------------------+
|patient_id|patient_name|     city|age|gender|membership|registration_date|membership_priority|
+----------+------------+---------+---+------+----------+-----------------+-------------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|                  1|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|                  2|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|                  1|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|                  3|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|                  2|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|                  1|
|      1007|Vikram Singh|  Chennai| 45|  Male|    Bronze|       2023-03-20|                  3|
|      1008|  Meera Nair|    Kochi| 28|F

In [0]:
visits_df.withColumn("visit_priority",when(visits_df.visit_status=="Pending","High").when(visits_df.visit_status=="Completed","Medium").otherwise("Low")).show()

+--------+----------+---------+----------+------------+-----------+---------+--------------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|test_cost|visit_priority|
+--------+----------+---------+----------+------------+-----------+---------+--------------+
|       1|      1001|      201|2024-03-01|   Completed|          2|     1000|        Medium|
|       2|      1002|      202|2024-03-01|   Completed|          1|      500|        Medium|
|       3|      1003|      203|2024-03-02|   Completed|          3|     1500|        Medium|
|       4|      1004|      204|2024-03-02|     Pending|          1|      500|          High|
|       5|      1005|      206|2024-03-03|   Completed|          2|     1000|        Medium|
|       6|      1006|      205|2024-03-03|   Completed|          4|     2000|        Medium|
|       7|      1007|      207|2024-03-04|   Cancelled|          1|      500|           Low|
|       8|      1008|      208|2024-03-04|   Completed|          2|   

In [0]:
payments_df.withColumn("high_bill_flag",when(payments_df.bill_amount>7500,"Yes").otherwise("No")).show()

+----------+--------+-----------+------------+--------------+--------------+
|payment_id|visit_id|bill_amount|payment_mode|payment_status|high_bill_flag|
+----------+--------+-----------+------------+--------------+--------------+
|       301|       1|       5200|         UPI|          Paid|            No|
|       302|       2|       2800| Credit Card|          Paid|            No|
|       303|       3|       7500|        Cash|          Paid|            No|
|       304|       4|       2900|         UPI|       Pending|            No|
|       305|       5|       5300|  Debit Card|          Paid|            No|
|       306|       6|      10000| Credit Card|          Paid|           Yes|
|       307|       7|       2850|        Cash|     Cancelled|            No|
|       308|       8|       5400|         UPI|          Paid|            No|
|       309|       9|       3200|         UPI|          Paid|            No|
|       310|      10|       4800| Credit Card|          Paid|            No|

In [0]:
patients_df = patients_df.withColumnRenamed("patient_name", "full_name")
patients_df.show()

+----------+------------+---------+---+------+----------+-----------------+
|patient_id|   full_name|     city|age|gender|membership|registration_date|
+----------+------------+---------+---+------+----------+-----------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|
|      1007|Vikram Singh|  Chennai| 45|  Male|    Bronze|       2023-03-20|
|      1008|  Meera Nair|    Kochi| 28|Female|    Silver|       2023-04-05|
|      1009|  Farhan Ali|Hyderabad| 39|  Male|      Gold|       2023-04-15|
|      1010| Divya Menon|Bengaluru| 33|Female|    Silver|       2023-04-21|
|      1011|

In [0]:
visits_df=visits_df.drop("test_cost")
visits_df.show()

+--------+----------+---------+----------+------------+-----------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|
+--------+----------+---------+----------+------------+-----------+
|       1|      1001|      201|2024-03-01|   Completed|          2|
|       2|      1002|      202|2024-03-01|   Completed|          1|
|       3|      1003|      203|2024-03-02|   Completed|          3|
|       4|      1004|      204|2024-03-02|     Pending|          1|
|       5|      1005|      206|2024-03-03|   Completed|          2|
|       6|      1006|      205|2024-03-03|   Completed|          4|
|       7|      1007|      207|2024-03-04|   Cancelled|          1|
|       8|      1008|      208|2024-03-04|   Completed|          2|
|       9|      1009|      201|2024-03-05|   Completed|          1|
|      10|      1010|      202|2024-03-05|   Completed|          2|
|      11|      1011|      205|2024-03-06|     Pending|          3|
|      12|      1012|      204|2024-03-06|   Com

---

# Part 3 — Joins

21. Join patients with visits.
22. Join visits with doctors.
23. Join visits with payments.
24. Create a final joined DataFrame containing patient, doctor, visit, and payment details.
25. Show patient name, city, doctor name, specialization, visit status, and bill amount.
26. Find visits where patient city and doctor city are different.
27. Find completed visits with paid payments.
28. Find pending visits with pending payments.
29. Find cancelled visits with cancelled payments.
30. Find patients who visited more than once.

---

In [0]:
patients_df.join(visits_df, "patient_id").show()

+----------+------------+---------+---+------+----------+-----------------+--------+---------+----------+------------+-----------+
|patient_id|   full_name|     city|age|gender|membership|registration_date|visit_id|doctor_id|visit_date|visit_status|tests_count|
+----------+------------+---------+---+------+----------+-----------------+--------+---------+----------+------------+-----------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|      21|      205|2024-03-11|   Completed|          3|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|       2|      202|2024-03-01|   Completed|          1|
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|       1|      201|2024-03-01|   Completed|          2|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|      22|      208|2024-03-11|   Completed|          2|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|       4

In [0]:
visits_df.join(
    payments_df,
    "visit_id"
).show()

+--------+----------+---------+----------+------------+-----------+----------+-----------+------------+--------------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|payment_id|bill_amount|payment_mode|payment_status|
+--------+----------+---------+----------+------------+-----------+----------+-----------+------------+--------------+
|       1|      1001|      201|2024-03-01|   Completed|          2|       301|       5200|         UPI|          Paid|
|       2|      1002|      202|2024-03-01|   Completed|          1|       302|       2800| Credit Card|          Paid|
|       3|      1003|      203|2024-03-02|   Completed|          3|       303|       7500|        Cash|          Paid|
|       4|      1004|      204|2024-03-02|     Pending|          1|       304|       2900|         UPI|       Pending|
|       5|      1005|      206|2024-03-03|   Completed|          2|       305|       5300|  Debit Card|          Paid|
|       6|      1006|      205|2024-03-03|   Com

In [0]:
final_df = patients_df.join(visits_df,"patient_id").join(doctors_df,
"doctor_id").join( payments_df,"visit_id")
final_df.show()

+--------+---------+----------+------------+---------+---+------+----------+-----------------+----------+------------+-----------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+
|visit_id|doctor_id|patient_id|   full_name|     city|age|gender|membership|registration_date|visit_date|visit_status|tests_count|     doctor_name|  specialization|doctor_city|consultation_fee|payment_id|bill_amount|payment_mode|payment_status|
+--------+---------+----------+------------+---------+---+------+----------+-----------------+----------+------------+-----------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+
|       1|      201|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|2024-03-01|   Completed|          2|Dr Sameer Sharma|      Cardiology|  Hyderabad|            1200|       301|       5200|         UPI|          Paid|
|       2|      202|

In [0]:
final_df.select("full_name","city","doctor_name","specialization","visit_status","bill_amount").show()

+------------+---------+----------------+----------------+------------+-----------+
|   full_name|     city|     doctor_name|  specialization|visit_status|bill_amount|
+------------+---------+----------------+----------------+------------+-----------+
|  Aarav Khan|Hyderabad|Dr Sameer Sharma|      Cardiology|   Completed|       5200|
| Priya Reddy|Bengaluru|  Dr Kavita Iyer|     Dermatology|   Completed|       2800|
| Rahul Mehta|   Mumbai|   Dr Imran Khan|     Orthopedics|   Completed|       7500|
|Sneha Kapoor|    Delhi| Dr Ramesh Reddy|      Pediatrics|     Pending|       2900|
| Kiran Patel|Ahmedabad|Dr Joseph Mathew|      Cardiology|   Completed|       5300|
|  Ananya Das|  Kolkata|  Dr Anita Mehta|       Neurology|   Completed|      10000|
|Vikram Singh|  Chennai|   Dr Fatima Ali|     Dermatology|   Cancelled|       2850|
|  Meera Nair|    Kochi|   Dr Arvind Rao|     Orthopedics|   Completed|       5400|
|  Farhan Ali|Hyderabad|Dr Sameer Sharma|      Cardiology|   Completed|     

In [0]:
final_df.filter(final_df.city!=final_df.doctor_city).show()

+--------+---------+----------+------------+---------+---+------+----------+-----------------+----------+------------+-----------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+
|visit_id|doctor_id|patient_id|   full_name|     city|age|gender|membership|registration_date|visit_date|visit_status|tests_count|     doctor_name|  specialization|doctor_city|consultation_fee|payment_id|bill_amount|payment_mode|payment_status|
+--------+---------+----------+------------+---------+---+------+----------+-----------------+----------+------------+-----------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+
|       5|      206|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|2024-03-03|   Completed|          2|Dr Joseph Mathew|      Cardiology|    Chennai|            1300|       305|       5300|  Debit Card|          Paid|
|       6|      205|

In [0]:
final_df.filter((final_df.visit_status=="Completed")&(final_df.payment_status=="Paid")).show()

+--------+---------+----------+------------+---------+---+------+----------+-----------------+----------+------------+-----------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+
|visit_id|doctor_id|patient_id|   full_name|     city|age|gender|membership|registration_date|visit_date|visit_status|tests_count|     doctor_name|  specialization|doctor_city|consultation_fee|payment_id|bill_amount|payment_mode|payment_status|
+--------+---------+----------+------------+---------+---+------+----------+-----------------+----------+------------+-----------+----------------+----------------+-----------+----------------+----------+-----------+------------+--------------+
|       1|      201|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|2024-03-01|   Completed|          2|Dr Sameer Sharma|      Cardiology|  Hyderabad|            1200|       301|       5200|         UPI|          Paid|
|       2|      202|

In [0]:
final_df.filter((final_df.visit_status=="Pending")&(final_df.payment_status=="Pending")).show()

+--------+---------+----------+------------+-------+---+------+----------+-----------------+----------+------------+-----------+----------------+--------------+-----------+----------------+----------+-----------+------------+--------------+
|visit_id|doctor_id|patient_id|   full_name|   city|age|gender|membership|registration_date|visit_date|visit_status|tests_count|     doctor_name|specialization|doctor_city|consultation_fee|payment_id|bill_amount|payment_mode|payment_status|
+--------+---------+----------+------------+-------+---+------+----------+-----------------+----------+------------+-----------+----------------+--------------+-----------+----------------+----------+-----------+------------+--------------+
|       4|      204|      1004|Sneha Kapoor|  Delhi| 26|Female|    Bronze|       2023-02-18|2024-03-02|     Pending|          1| Dr Ramesh Reddy|    Pediatrics|      Delhi|             900|       304|       2900|         UPI|       Pending|
|      11|      205|      1011|  Arj

In [0]:
final_df.filter((final_df.visit_status=="Cancelled")&(final_df.payment_status=="Cancelled")).show()

+--------+---------+----------+------------+-------+---+------+----------+-----------------+----------+------------+-----------+-------------+--------------+-----------+----------------+----------+-----------+------------+--------------+
|visit_id|doctor_id|patient_id|   full_name|   city|age|gender|membership|registration_date|visit_date|visit_status|tests_count|  doctor_name|specialization|doctor_city|consultation_fee|payment_id|bill_amount|payment_mode|payment_status|
+--------+---------+----------+------------+-------+---+------+----------+-----------------+----------+------------+-----------+-------------+--------------+-----------+----------------+----------+-----------+------------+--------------+
|      16|      207|      1016| Ayesha Khan|Kolkata| 27|Female|    Bronze|       2023-06-20|2024-03-08|   Cancelled|          2|Dr Fatima Ali|   Dermatology|    Kolkata|             850|       316|       4850|        Cash|     Cancelled|
|       7|      207|      1007|Vikram Singh|Chen

In [0]:
final_df.groupBy(
    "full_name"
).count().filter(
    col("count") > 1
).show()

+------------+-----+
|   full_name|count|
+------------+-----+
| Rahul Mehta|    2|
|  Aarav Khan|    2|
|  Ananya Das|    2|
|  Farhan Ali|    2|
|Kavya Sharma|    2|
+------------+-----+



---

# Part 4 — Aggregations

31. Count patients by city.
32. Count patients by membership.
33. Count doctors by specialization.
34. Count visits by status.
35. Count payments by payment mode.
36. Calculate total bill amount.
37. Calculate average bill amount.
38. Calculate total revenue by city.
39. Calculate total revenue by specialization.
40. Calculate total revenue by doctor.

---

In [0]:
patients_df.groupBy(patients_df.city).count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Hyderabad|    3|
|Bengaluru|    2|
|   Mumbai|    2|
|    Delhi|    2|
|Ahmedabad|    2|
|  Kolkata|    2|
|  Chennai|    3|
|    Kochi|    2|
|     Pune|    1|
|  Lucknow|    1|
+---------+-----+



In [0]:
patients_df.groupBy(patients_df.membership).count().show()

+----------+-----+
|membership|count|
+----------+-----+
|      Gold|    8|
|    Silver|    8|
|    Bronze|    4|
+----------+-----+



In [0]:
doctors_df.groupBy(doctors_df.specialization).count().show()

+----------------+-----+
|  specialization|count|
+----------------+-----+
|      Cardiology|    2|
|     Dermatology|    2|
|     Orthopedics|    2|
|      Pediatrics|    1|
|       Neurology|    2|
|General Medicine|    1|
+----------------+-----+



In [0]:
visits_df.groupBy("visit_status").count().show()

+------------+-----+
|visit_status|count|
+------------+-----+
|   Completed|   20|
|     Pending|    3|
|   Cancelled|    2|
+------------+-----+



In [0]:
payments_df.groupBy("payment_mode").count().show()

+------------+-----+
|payment_mode|count|
+------------+-----+
|         UPI|   11|
| Credit Card|    6|
|        Cash|    5|
|  Debit Card|    3|
+------------+-----+



In [0]:
payments_df.agg(sum("bill_amount").alias("total_bill"),avg("bill_amount").alias("average_bill")).show()

+----------+------------+
|total_bill|average_bill|
+----------+------------+
|    133600|      5344.0|
+----------+------------+



In [0]:
final_df.groupBy("city").agg(sum("bill_amount").alias("total_revenue")).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Hyderabad|        31100|
|   Mumbai|        18400|
|Bengaluru|         7600|
|    Delhi|         5800|
|Ahmedabad|        10600|
|  Kolkata|        18050|
|  Chennai|        16150|
|    Kochi|        13300|
|     Pune|         2700|
|  Lucknow|         9900|
+---------+-------------+



In [0]:
final_df.groupBy("specialization").agg(sum("bill_amount").alias("total_revenue")).show()

+----------------+-------------+
|  specialization|total_revenue|
+----------------+-------------+
|     Orthopedics|        23800|
|      Cardiology|        34700|
|     Dermatology|        18100|
|      Pediatrics|         5800|
|       Neurology|        43800|
|General Medicine|         7400|
+----------------+-------------+



In [0]:
final_df.groupBy("doctor_name").agg(sum("bill_amount").alias("total_revenue")).show()

+----------------+-------------+
|     doctor_name|total_revenue|
+----------------+-------------+
|   Dr Imran Khan|        13000|
|  Dr Kavita Iyer|        10400|
|Dr Sameer Sharma|        18800|
|  Dr Anita Mehta|        26000|
|Dr Joseph Mathew|        15900|
| Dr Ramesh Reddy|         5800|
|   Dr Arvind Rao|        10800|
|   Dr Fatima Ali|         7700|
| Dr Ganesh Patil|         7400|
|   Dr Leela Nair|        17800|
+----------------+-------------+



---

# Part 5 — Spark SQL

41. Create temp views for all DataFrames.
42. Show all patients using SQL.
43. Find Cardiology visits using SQL.
44. Find revenue by city using SQL.
45. Find revenue by specialization using SQL.
46. Find top 5 highest bill visits.
47. Count visits per doctor.
48. Count visits per patient.
49. Find average bill by payment mode.
50. Find patients with total billing above 10000.

---

In [0]:
patients_df.createOrReplaceTempView("patients")

doctors_df.createOrReplaceTempView("doctors")

visits_df.createOrReplaceTempView("visits")

payments_df.createOrReplaceTempView("payments")

In [0]:
spark.sql("select * from patients").show()

+----------+------------+---------+---+------+----------+-----------------+
|patient_id|   full_name|     city|age|gender|membership|registration_date|
+----------+------------+---------+---+------+----------+-----------------+
|      1001|  Aarav Khan|Hyderabad| 29|  Male|      Gold|       2023-01-10|
|      1002| Priya Reddy|Bengaluru| 34|Female|    Silver|       2023-01-15|
|      1003| Rahul Mehta|   Mumbai| 41|  Male|      Gold|       2023-02-02|
|      1004|Sneha Kapoor|    Delhi| 26|Female|    Bronze|       2023-02-18|
|      1005| Kiran Patel|Ahmedabad| 37|  Male|    Silver|       2023-03-01|
|      1006|  Ananya Das|  Kolkata| 31|Female|      Gold|       2023-03-12|
|      1007|Vikram Singh|  Chennai| 45|  Male|    Bronze|       2023-03-20|
|      1008|  Meera Nair|    Kochi| 28|Female|    Silver|       2023-04-05|
|      1009|  Farhan Ali|Hyderabad| 39|  Male|      Gold|       2023-04-15|
|      1010| Divya Menon|Bengaluru| 33|Female|    Silver|       2023-04-21|
|      1011|

In [0]:
spark.sql("select * from visits v join doctors d on v.doctor_id = d.doctor_id where specialization = 'Cardiology'").show()


+--------+----------+---------+----------+------------+-----------+---------+----------------+--------------+-----------+----------------+
|visit_id|patient_id|doctor_id|visit_date|visit_status|tests_count|doctor_id|     doctor_name|specialization|doctor_city|consultation_fee|
+--------+----------+---------+----------+------------+-----------+---------+----------------+--------------+-----------+----------------+
|       1|      1001|      201|2024-03-01|   Completed|          2|      201|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|
|       5|      1005|      206|2024-03-03|   Completed|          2|      206|Dr Joseph Mathew|    Cardiology|    Chennai|            1300|
|       9|      1009|      201|2024-03-05|   Completed|          1|      201|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|
|      14|      1014|      201|2024-03-07|   Completed|          3|      201|Dr Sameer Sharma|    Cardiology|  Hyderabad|            1200|
|      18|      1018|      

In [0]:
spark.sql("select p.city,sum(pay.bill_amount) as revenue from patients p join visits v on p.patient_id = v.patient_id join payments pay on v.visit_id = pay.visit_id group by p.city").show()

+---------+-------+
|     city|revenue|
+---------+-------+
|Hyderabad|  31100|
|   Mumbai|  18400|
|Bengaluru|   7600|
|    Delhi|   5800|
|Ahmedabad|  10600|
|  Kolkata|  18050|
|  Chennai|  16150|
|    Kochi|  13300|
|     Pune|   2700|
|  Lucknow|   9900|
+---------+-------+



In [0]:
spark.sql("select d.specialization,sum(pay.bill_amount) as revenue from doctors d join visits v on d.doctor_id = v.doctor_id join payments pay on v.visit_id = pay.visit_id group by d.specialization").show()

+----------------+-------+
|  specialization|revenue|
+----------------+-------+
|     Orthopedics|  23800|
|      Cardiology|  34700|
|     Dermatology|  18100|
|      Pediatrics|   5800|
|       Neurology|  43800|
|General Medicine|   7400|
+----------------+-------+



In [0]:
spark.sql("select * from payments order by bill_amount desc limit 5").show()

+----------+--------+-----------+------------+--------------+
|payment_id|visit_id|bill_amount|payment_mode|payment_status|
+----------+--------+-----------+------------+--------------+
|       306|       6|      10000| Credit Card|          Paid|
|       317|      17|       9900| Credit Card|          Paid|
|       321|      21|       8000|         UPI|          Paid|
|       311|      11|       8000|         UPI|       Pending|
|       319|      19|       7900|  Debit Card|          Paid|
+----------+--------+-----------+------------+--------------+



In [0]:
spark.sql("select doctor_id,count(*) as total_visits from visits group by doctor_id").show()



+---------+------------+
|doctor_id|total_visits|
+---------+------------+
|      201|           4|
|      202|           3|
|      203|           2|
|      204|           2|
|      206|           3|
|      205|           3|
|      207|           2|
|      208|           2|
|      210|           2|
|      209|           2|
+---------+------------+



In [0]:
spark.sql("select patient_id,count(*) as total_visits from visits group by patient_id").show()

+----------+------------+
|patient_id|total_visits|
+----------+------------+
|      1001|           2|
|      1002|           1|
|      1003|           2|
|      1004|           1|
|      1005|           1|
|      1006|           2|
|      1007|           1|
|      1008|           1|
|      1009|           2|
|      1010|           1|
|      1011|           1|
|      1012|           1|
|      1013|           1|
|      1014|           2|
|      1015|           1|
|      1016|           1|
|      1017|           1|
|      1018|           1|
|      1019|           1|
|      1020|           1|
+----------+------------+



In [0]:
spark.sql("select payment_mode,avg(bill_amount) as average_bill from payments group by payment_mode").show()

+------------+-----------------+
|payment_mode|     average_bill|
+------------+-----------------+
|         UPI|5272.727272727273|
| Credit Card|           6400.0|
|        Cash|           4260.0|
|  Debit Card|           5300.0|
+------------+-----------------+



In [0]:
spark.sql("select p.full_name,sum(pay.bill_amount) as total_bill from patients p join visits v on p.patient_id = v.patient_id join payments pay on v.visit_id = pay.visit_id group by p.full_name having sum(pay.bill_amount) > 10000").show()

+-----------+----------+
|  full_name|total_bill|
+-----------+----------+
|Rahul Mehta|     12900|
| Aarav Khan|     13200|
| Ananya Das|     13200|
+-----------+----------+



---

# Part 6 — Window Functions

51. Rank patients by total bill within city.
52. Rank doctors by total revenue within specialization.
53. Use ROW_NUMBER to find top billing patient per city.
54. Use DENSE_RANK to rank doctors by consultation fee within specialization.
55. Find top 2 doctors by revenue.
56. Create running total revenue by visit date.
57. Create running total revenue by doctor.
58. Rank cities by total revenue.
59. Rank specializations by total revenue.
60. Find highest bill visit per payment mode.

---

In [0]:
from pyspark.sql import Window

In [0]:

patient_bill_df = final_df.groupBy("full_name","city").agg(sum("bill_amount").alias("total_bill"))

window1 = Window.partitionBy("city").orderBy(col("total_bill").desc())

patient_bill_df.withColumn("rank",rank().over(window1)).show()

+------------+---------+----------+----+
|   full_name|     city|total_bill|rank|
+------------+---------+----------+----+
| Kiran Patel|Ahmedabad|      5300|   1|
|  Pooja Shah|Ahmedabad|      5300|   1|
| Divya Menon|Bengaluru|      4800|   1|
| Priya Reddy|Bengaluru|      2800|   2|
|  Arjun Iyer|  Chennai|      8000|   1|
| Lakshmi Rao|  Chennai|      5300|   2|
|Vikram Singh|  Chennai|      2850|   3|
|Sneha Kapoor|    Delhi|      2900|   1|
|  Neha Gupta|    Delhi|      2900|   1|
|  Aarav Khan|Hyderabad|     13200|   1|
|Kavya Sharma|Hyderabad|     10000|   2|
|  Farhan Ali|Hyderabad|      7900|   3|
|  Rohan Nair|    Kochi|      7900|   1|
|  Meera Nair|    Kochi|      5400|   2|
|  Ananya Das|  Kolkata|     13200|   1|
| Ayesha Khan|  Kolkata|      4850|   2|
|Manish Yadav|  Lucknow|      9900|   1|
| Rahul Mehta|   Mumbai|     12900|   1|
|  Sanjay Rao|   Mumbai|      5500|   2|
|Nikhil Verma|     Pune|      2700|   1|
+------------+---------+----------+----+



In [0]:
doctor_revenue_df = final_df.groupBy("doctor_name","specialization").agg(sum("bill_amount").alias("revenue"))

window2 = Window.partitionBy("specialization").orderBy(col("revenue").desc())

doctor_revenue_df.withColumn("rank",rank().over(window2)).show()

+----------------+----------------+-------+----+
|     doctor_name|  specialization|revenue|rank|
+----------------+----------------+-------+----+
|Dr Sameer Sharma|      Cardiology|  18800|   1|
|Dr Joseph Mathew|      Cardiology|  15900|   2|
|  Dr Kavita Iyer|     Dermatology|  10400|   1|
|   Dr Fatima Ali|     Dermatology|   7700|   2|
| Dr Ganesh Patil|General Medicine|   7400|   1|
|  Dr Anita Mehta|       Neurology|  26000|   1|
|   Dr Leela Nair|       Neurology|  17800|   2|
|   Dr Imran Khan|     Orthopedics|  13000|   1|
|   Dr Arvind Rao|     Orthopedics|  10800|   2|
| Dr Ramesh Reddy|      Pediatrics|   5800|   1|
+----------------+----------------+-------+----+



In [0]:
patient_bill_df.withColumn("row_num",row_number().over(window1)) \
.filter(col("row_num") == 1).show()

+------------+---------+----------+-------+
|   full_name|     city|total_bill|row_num|
+------------+---------+----------+-------+
| Kiran Patel|Ahmedabad|      5300|      1|
| Divya Menon|Bengaluru|      4800|      1|
|  Arjun Iyer|  Chennai|      8000|      1|
|Sneha Kapoor|    Delhi|      2900|      1|
|  Aarav Khan|Hyderabad|     13200|      1|
|  Rohan Nair|    Kochi|      7900|      1|
|  Ananya Das|  Kolkata|     13200|      1|
|Manish Yadav|  Lucknow|      9900|      1|
| Rahul Mehta|   Mumbai|     12900|      1|
|Nikhil Verma|     Pune|      2700|      1|
+------------+---------+----------+-------+



In [0]:

window3 = Window.partitionBy("specialization").orderBy(col("consultation_fee").desc())

doctors_df.withColumn("dense_rank",dense_rank().over(window3)).show()

+---------+----------------+----------------+-----------+----------------+----------+
|doctor_id|     doctor_name|  specialization|doctor_city|consultation_fee|dense_rank|
+---------+----------------+----------------+-----------+----------------+----------+
|      206|Dr Joseph Mathew|      Cardiology|    Chennai|            1300|         1|
|      201|Dr Sameer Sharma|      Cardiology|  Hyderabad|            1200|         2|
|      207|   Dr Fatima Ali|     Dermatology|    Kolkata|             850|         1|
|      202|  Dr Kavita Iyer|     Dermatology|  Bengaluru|             800|         2|
|      210| Dr Ganesh Patil|General Medicine|       Pune|             700|         1|
|      205|  Dr Anita Mehta|       Neurology|  Hyderabad|            2000|         1|
|      209|   Dr Leela Nair|       Neurology|      Kochi|            1900|         2|
|      203|   Dr Imran Khan|     Orthopedics|     Mumbai|            1500|         1|
|      208|   Dr Arvind Rao|     Orthopedics|  Bengalu

In [0]:
window4 = Window.orderBy(col("revenue").desc())

doctor_revenue_df.withColumn("rank",rank().over(window4)) \
.filter(col("rank") <= 2).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------------+--------------+-------+----+
|     doctor_name|specialization|revenue|rank|
+----------------+--------------+-------+----+
|  Dr Anita Mehta|     Neurology|  26000|   1|
|Dr Sameer Sharma|    Cardiology|  18800|   2|
+----------------+--------------+-------+----+



In [0]:
daily_revenue_df = final_df.groupBy("visit_date").agg(sum("bill_amount").alias("daily_revenue"))

window5 = Window.orderBy("visit_date")

daily_revenue_df.withColumn("running_total",sum("daily_revenue").over(window5)).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------+-------------+-------------+
|visit_date|daily_revenue|running_total|
+----------+-------------+-------------+
|2024-03-01|         8000|         8000|
|2024-03-02|        10400|        18400|
|2024-03-03|        15300|        33700|
|2024-03-04|         8250|        41950|
|2024-03-05|         8000|        49950|
|2024-03-06|        10900|        60850|
|2024-03-07|        12700|        73550|
|2024-03-08|         7550|        81100|
|2024-03-09|        15200|        96300|
|2024-03-10|        13200|       109500|
|2024-03-11|        13400|       122900|
|2024-03-12|         7900|       130800|
|2024-03-13|         2800|       133600|
+----------+-------------+-------------+



In [0]:
doctor_daily_df = final_df.groupBy("doctor_name").agg(sum("bill_amount").alias("doctor_revenue"))

window6 = Window.orderBy("doctor_revenue")

doctor_daily_df.withColumn("running_total",sum("doctor_revenue").over(window6)).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------------+--------------+-------------+
|     doctor_name|doctor_revenue|running_total|
+----------------+--------------+-------------+
| Dr Ramesh Reddy|          5800|         5800|
| Dr Ganesh Patil|          7400|        13200|
|   Dr Fatima Ali|          7700|        20900|
|  Dr Kavita Iyer|         10400|        31300|
|   Dr Arvind Rao|         10800|        42100|
|   Dr Imran Khan|         13000|        55100|
|Dr Joseph Mathew|         15900|        71000|
|   Dr Leela Nair|         17800|        88800|
|Dr Sameer Sharma|         18800|       107600|
|  Dr Anita Mehta|         26000|       133600|
+----------------+--------------+-------------+



In [0]:
city_revenue_df = final_df.groupBy("city").agg(sum("bill_amount").alias("revenue"))

window7 = Window.orderBy(col("revenue").desc())

city_revenue_df.withColumn("rank",rank().over(window7)).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-------+----+
|     city|revenue|rank|
+---------+-------+----+
|Hyderabad|  31100|   1|
|   Mumbai|  18400|   2|
|  Kolkata|  18050|   3|
|  Chennai|  16150|   4|
|    Kochi|  13300|   5|
|Ahmedabad|  10600|   6|
|  Lucknow|   9900|   7|
|Bengaluru|   7600|   8|
|    Delhi|   5800|   9|
|     Pune|   2700|  10|
+---------+-------+----+



In [0]:
spec_revenue_df = final_df.groupBy("specialization").agg(sum("bill_amount").alias("revenue"))

window8 = Window.orderBy(col("revenue").desc())

spec_revenue_df.withColumn("rank",rank().over(window8)).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----------------+-------+----+
|  specialization|revenue|rank|
+----------------+-------+----+
|       Neurology|  43800|   1|
|      Cardiology|  34700|   2|
|     Orthopedics|  23800|   3|
|     Dermatology|  18100|   4|
|General Medicine|   7400|   5|
|      Pediatrics|   5800|   6|
+----------------+-------+----+



In [0]:
window9 = Window.partitionBy("payment_mode").orderBy(col("bill_amount").desc())

payments_df.withColumn("row_num",row_number().over(window9)) \
.filter(col("row_num") == 1).show()

+----------+--------+-----------+------------+--------------+-------+
|payment_id|visit_id|bill_amount|payment_mode|payment_status|row_num|
+----------+--------+-----------+------------+--------------+-------+
|       303|       3|       7500|        Cash|          Paid|      1|
|       306|       6|      10000| Credit Card|          Paid|      1|
|       319|      19|       7900|  Debit Card|          Paid|      1|
|       311|      11|       8000|         UPI|       Pending|      1|
+----------+--------+-----------+------------+--------------+-------+



---

# Part 7 — Delta Lake Core

61. Save final joined data as a Delta table.
62. Insert two new visit records.
63. Update visit status for one pending visit.
64. Update payment status for one pending payment.
65. Delete cancelled visits from a Delta table.
66. Create a separate Delta table for cleaned completed visits.
67. Query Delta table history.
68. Query an older version using time travel.
69. Compare latest version and older version.
70. Run VACUUM DRY RUN.

---

In [0]:
final_df.write.format("delta").mode("overwrite").saveAsTable("finaldataq")

In [0]:
%sql
insert into finaldataq values
(101, 201, 1101, 'Test Patient1', 'Bengaluru', 30, 'Male', 'Gold', '2024-01-01', '2024-04-01', 'Pending', 2, 'Dr Sameer Sharma', 'Cardiology', 'Hyderabad', 1200, 401, 5000, 'UPI', 'Pending'),

(102, 202, 1102, 'Test Patient2', 'Chennai', 28, 'Female', 'Silver', '2024-01-05', '2024-04-02', 'Completed', 1, 'Dr Kavita Iyer', 'Dermatology', 'Bengaluru', 800, 402, 3000, 'Cash', 'Paid');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql 
update finaldataq set visit_status="Completed" where visit_status="Pending";

num_affected_rows
4


In [0]:
%sql
update finaldataq set payment_status="Paid" where payment_status="Pending";

num_affected_rows
4


In [0]:
%sql
delete from finaldataq where visit_status="Cancelled";

num_affected_rows
2


In [0]:
%sql 
create or replace table cleaned_visits using delta as select * from finaldataq where visit_status="Completed";

num_affected_rows,num_inserted_rows


In [0]:
%sql
describe history finaldataq;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-05-06T08:41:40.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(visit_status#19398 = Cancelled)""])",null,List(897402096670738),feb46989-3698-41d5-b929-cb62fb523acd,0506-083246-bwuppeke-v2n,5,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1429, numDeletionVectorsUpdated -> 0, numDeletedRows -> 2, scanTimeMs -> 1049, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 374)",null,Databricks-Runtime/18.1.x-photon-scala2.13
5,2026-05-06T08:41:15.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(897402096670738),da0aceaa-60f4-434c-9463-d8356ef59bd3,0506-083246-bwuppeke-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 2, numRemovedBytes -> 12851, p25FileSize -> 7121, numDeletionVectorsRemoved -> 1, minFileSize -> 7121, numAddedFiles -> 1, maxFileSize -> 7121, p75FileSize -> 7121, p50FileSize -> 7121, numAddedBytes -> 7121)",null,Databricks-Runtime/18.1.x-photon-scala2.13
4,2026-05-06T08:41:14.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(payment_status#18287 = Pending)""])",null,List(897402096670738),da0aceaa-60f4-434c-9463-d8356ef59bd3,0506-083246-bwuppeke-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 2262, numDeletionVectorsUpdated -> 0, scanTimeMs -> 879, numAddedFiles -> 1, numUpdatedRows -> 4, numAddedBytes -> 5730, rewriteTimeMs -> 1383)",null,Databricks-Runtime/18.1.x-photon-scala2.13
3,2026-05-06T08:40:51.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(897402096670738),1eb7199c-84b0-4ba6-80be-6e335f5a64d3,0506-083246-bwuppeke-v2n,2,SnapshotIsolation,false,"Map(numRemovedFiles -> 10, numRemovedBytes -> 55482, p25FileSize -> 7121, numDeletionVectorsRemoved -> 4, minFileSize -> 7121, numAddedFiles -> 1, maxFileSize -> 7121, p75FileSize -> 7121, p50FileSize -> 7121, numAddedBytes -> 7121)",null,Databricks-Runtime/18.1.x-photon-scala2.13
2,2026-05-06T08:40:48.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(visit_status#17132 = Pending)""])",null,List(897402096670738),1eb7199c-84b0-4ba6-80be-6e335f5a64d3,0506-083246-bwuppeke-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 4, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4467, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2347, numAddedFiles -> 1, numUpdatedRows -> 4, numAddedBytes -> 5745, rewriteTimeMs -> 2077)",null,Databricks-Runtime/18.1.x-photon-scala2.13
1,2026-05-06T08:37:40.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(897402096670738),b0bda787-69de-44ae-b652-b542a1a8ddc4,0506-083246-bwuppeke-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 5131)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-05-06T08:34:02.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDel

In [0]:
%sql
select * from finaldataq version as of 0

visit_id,doctor_id,patient_id,full_name,city,age,gender,membership,registration_date,visit_date,visit_status,tests_count,doctor_name,specialization,doctor_city,consultation_fee,payment_id,bill_amount,payment_mode,payment_status
22,208,1003,Rahul Mehta,Mumbai,41,Male,Gold,2023-02-02,2024-03-11,Completed,2,Dr Arvind Rao,Orthopedics,Bengaluru,1400,322,5400,Credit Card,Paid
23,201,1006,Ananya Das,Kolkata,31,Female,Gold,2023-03-12,2024-03-12,Completed,1,Dr Sameer Sharma,Cardiology,Hyderabad,1200,323,3200,Cash,Paid
24,210,1009,Farhan Ali,Hyderabad,39,Male,Gold,2023-04-15,2024-03-12,Completed,2,Dr Ganesh Patil,General Medicine,Pune,700,324,4700,UPI,Paid
25,202,1014,Kavya Sharma,Hyderabad,30,Female,Gold,2023-06-01,2024-03-13,Completed,1,Dr Kavita Iyer,Dermatology,Bengaluru,800,325,2800,UPI,Paid
19,209,1019,Rohan Nair,Kochi,40,Male,Gold,2023-08-01,2024-03-10,Completed,3,Dr Leela Nair,Neurology,Kochi,1900,319,7900,Debit Card,Paid
20,206,1020,Lakshmi Rao,Chennai,35,Female,Silver,2023-08-14,2024-03-10,Pending,2,Dr Joseph Mathew,Cardiology,Chennai,1300,320,5300,UPI,Pending
21,205,1001,Aarav Khan,Hyderabad,29,Male,Gold,2023-01-10,2024-03-11,Completed,3,Dr Anita Mehta,Neurology,Hyderabad,2000,321,8000,UPI,Paid
7,207,1007,Vikram Singh,Chennai,45,Male,Bronze,2023-03-20,2024-03-04,Cancelled,1,Dr Fatima Ali,Dermatology,Kolkata,850,307,2850,Cash,Cancelled
8,208,1008,Meera Nair,Kochi,28,Female,Silver,2023-04-05,2024-03-04,Completed,2,Dr Arvind Rao,Orthopedics,Bengaluru,1400,308,5400,UPI,Paid
9,201,1009,Farhan Ali,Hyderabad,39,Male,Gold,2023-04-15,2024-03-05,Completed,1,Dr Sameer Sharma,Cardiology,Hyderabad,1200,309,3200,UPI,Paid


In [0]:
%sql
VACUUM finaldataq RETAIN 168 HOURS DRY RUN;

path


---

# Part 8 — Merge / Upsert

```python
daily_visits_data = [

(26,1002,201,"2024-03-14","Completed",2),

(27,1005,210,"2024-03-14","Completed",1),

(11,1011,205,"2024-03-06","Completed",3),

(20,1020,206,"2024-03-10","Completed",2),

(28,1018,203,"2024-03-15","Pending",2)

]

daily_visits_columns = [

"visit_id","patient_id","doctor_id","visit_date","visit_status","tests_count"

]

daily_visits_df = spark.createDataFrame(daily_visits_data, daily_visits_columns)
```

71. Create a Delta target table for visits.
72. Load initial visits into target table.
73. Create a temp view for daily updates.
74. Merge daily updates into the target table.
75. Existing visits should be updated.
76. New visits should be inserted.
77. Verify updated visit IDs.
78. Verify newly inserted visit IDs.
79. Check Delta history after merge.
80. Explain why MERGE is used for incremental loads.

---


In [0]:
daily_visits_data = [

(26,1002,201,"2024-03-14","Completed",2),

(27,1005,210,"2024-03-14","Completed",1),

(11,1011,205,"2024-03-06","Completed",3),

(20,1020,206,"2024-03-10","Completed",2),

(28,1018,203,"2024-03-15","Pending",2)

]

daily_visits_columns = [

"visit_id","patient_id","doctor_id","visit_date","visit_status","tests_count"

]

daily_visits_df = spark.createDataFrame(daily_visits_data, daily_visits_columns)

In [0]:
daily_visits_df.write.format("delta").mode("overwrite").saveAsTable("daily_visits")

In [0]:
%sql
select * from daily_visits;

visit_id,patient_id,doctor_id,visit_date,visit_status,tests_count
26,1002,201,2024-03-14,Completed,2
27,1005,210,2024-03-14,Completed,1
11,1011,205,2024-03-06,Completed,3
20,1020,206,2024-03-10,Completed,2
28,1018,203,2024-03-15,Pending,2


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW source_visits AS
SELECT * FROM VALUES
(26, 1002, 201, '2024-03-14', 'Completed', 2),
(27, 1005, 210, '2024-03-14', 'Completed', 1),
(11, 1011, 205, '2024-03-06', 'Completed', 3),
(20, 1020, 206, '2024-03-10', 'Completed', 2),
(28, 1018, 203, '2024-03-15', 'Pending', 2)
AS source_visits(visit_id,patient_id,doctor_id,visit_date,visit_status,tests_count);

In [0]:
%sql
MERGE INTO daily_visits AS target
USING source_visits AS source
ON target.visit_id = source.visit_id

WHEN MATCHED THEN
UPDATE SET
    target.patient_id = source.patient_id,
    target.doctor_id = source.doctor_id,
    target.visit_date = source.visit_date,
    target.visit_status = source.visit_status,
    target.tests_count = source.tests_count

WHEN NOT MATCHED THEN
INSERT (
    visit_id,
    patient_id,
    doctor_id,
    visit_date,
    visit_status,
    tests_count
)
VALUES (
    source.visit_id,
    source.patient_id,
    source.doctor_id,
    source.visit_date,
    source.visit_status,
    source.tests_count
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
%sql
describe history daily_visits;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-05-06T08:52:47.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#23064L = cast(visit_id#23052 as bigint))""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(897402096670738),6a9b150d-e086-4337-9cae-21010273e125,0506-083246-bwuppeke-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 5, numTargetBytesAdded -> 9112, numTargetBytesRemoved -> 2049, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 2999, materializeSourceTimeMs -> 327, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1323, numTargetRowsUpdated -> 5, numOutputRows -> 5, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1282)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-05-06T08:49:18.000Z,144220425958109,azuser5811_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(897402096670738),418bcf0c-b2f8-4b2d-99cb-1c5e91d95afc,0506-083246-bwuppeke-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 5, numOutputBytes -> 2049)",null,Databricks-Runtime/18.1.x-photon-scala2.13
